# A · Official train + val · seen-val

Запустите этот notebook на отдельной RTX 5090. Откройте его из распакованной папки и выберите **Run All**. **Seen-val.** В A и common dev, и официальный val входят в fit. Смотрите эти числа только как диагностику обучения.

На подготовку модели, CUDA smoke, обучение, оба инференса и упаковку выделено 150 минут. Установка окружения и последующая CPU-оценка идут отдельно. Лимит может остановить обучение или генерацию. Прерванные звонки учитываются в метриках.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import zipfile

ROOT = Path.cwd().resolve()
CONFIG = json.loads((ROOT / 'trial_config.json').read_text(encoding='utf-8'))
assert CONFIG['trial_id'] == 'A_official_train_val_r32', 'Откройте ноутбук из его папки'
RUN_NAME = 'run01'
MODE = 'train'  # check — токенизация, smoke — CUDA + longest fit/dev, train — весь опыт
VENV = ROOT / '.venv312'
PYTHON = VENV / 'bin/python'
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false')


## Подготовка окружения

Требуется доступный `python3.12` с модулем venv. Если его нет в образе Clore, см. INSTRUCTION_RU.md.

In [ ]:
if not PYTHON.exists():
    py312 = shutil.which('python3.12')
    if not py312:
        raise RuntimeError('В образе Clore требуется python3.12. См. INSTRUCTION_RU.md.')
    subprocess.run([py312, '-m', 'venv', str(VENV)], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', 'torch==2.7.1',
                '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-r', str(ROOT/'code/requirements.txt')], check=True)


## Запуск 150-минутного опыта

Preflight проверяет полные диалоги и assistant-only loss. Smoke обучает на самом длинном fit-примере и оценивает самый длинный dev-пример.

In [ ]:
cmd = [str(PYTHON), '-B', str(ROOT/'code/worker.py'), '--root', str(ROOT),
       '--run-name', RUN_NAME, '--mode', MODE]
subprocess.run(cmd, cwd=ROOT, env=env, check=True)


## Подсчёт метрик по сохранённым ответам

Отдельный CPU-процесс использует официальный scorer 2.1. Этот же расчёт можно выполнить локально с RESULT.zip и отдельным архивом оценщика.

In [ ]:
import hashlib
from IPython.display import FileLink, display

if MODE == 'train':
    output = ROOT / f"{CONFIG['trial_id']}_{RUN_NAME}_RESULT.zip"
    display(FileLink(output.name))
    archive = ROOT/'CPU_EVALUATOR.zip'
    expected = json.loads((ROOT/'DELIVERY.json').read_text())['evaluator_zip_sha256']
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected
    extracts = ROOT / 'runs' / RUN_NAME / 'evaluator'
    extracts.mkdir(exist_ok=False)
    with zipfile.ZipFile(archive) as z:
        z.extractall(extracts)
    evaluator = extracts / 'qwen06_evaluator'
    score_dir = ROOT / f"{CONFIG['trial_id']}_{RUN_NAME}_METRICS"
    subprocess.run([str(PYTHON), '-B', str(evaluator/'code/score_one.py'),
                    '--result-zip', str(output), '--coordinator', str(evaluator),
                    '--out', str(score_dir)], cwd=ROOT, check=True)
    display(FileLink(score_dir.with_suffix('.zip').name))
    display(FileLink(str(score_dir.relative_to(ROOT)/'REPORT_RU.md')))
